<a href="https://colab.research.google.com/github/busycaesar/GPT/blob/Master/main-script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# Hyperparameters
# Parallel process on GPU
batch_size = 32
# Maximum context length for predicting next token
block_size = 8
# Iterations for training the model
max_iters = 3000
# Evaluate the loss while training after regular interval
eval_interval = 300
# Learning rate: controls how much weights change at every update step
learning_rate = 1e-2
# Use GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200

# To maintain reproducability of random indices.
torch.manual_seed(1337)

# Download the dataset to train on.
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# All the unique characters that occur in the text
unique_characters = sorted(list(set(text)))
vocab_size = len(unique_characters)

# Mapping for each unique character.
stoi = { ch:i for i,ch in enumerate(unique_characters) }
itos = { i:ch for i,ch in enumerate(unique_characters) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

encoded_text = encode(text)

dataset = torch.tensor(encoded_text, dtype=torch.long)

# Split the data into train and validation sets
train_dataset_proportion = 0.9
number_of_dataset = int(train_dataset_proportion*len(dataset))

train_dataset = dataset[:number_of_dataset]
validation_dataset = dataset[number_of_dataset:]

def get_batch(split):
    dataset = train_dataset if split == 'train' else validation_dataset
    # Returns "batch_size" (4) random starting indices from the dataset.
    # The upper bound is "len(dataset) - block_size" (1003854 - 8) so that there are enough tokens for the block size even if the largest possible index is picked.
    ix = torch.randint(len(dataset) - block_size, (batch_size,))

    # Get "block_size" tokens starting at each chosen index, for all indices.
    context = torch.stack([dataset[i:i+block_size] for i in ix])

    # Get "block_size" tokens starting one position after each chosen index, for all indices.
    target = torch.stack([dataset[i+1:i+block_size+1] for i in ix])
    return context, target

################################################################################
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out
################################################################################

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Lookup table mapping each token id to a row of vocab_size numbers. Each cell is assigned random values before training.
        # Normally, each token id is mapped to the row that contains the embedding (carrying semantic meaning) of the token.
        # The embeddings are then converted into logits to predict the next token.
        # Here, we skip all of that. The row length already equals vocab_size, so the row can be used directly as the logits.
        # So this model learns the logits directly in the table, instead of learning the many layers of weights that a real model uses to produce them.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, context_ids, targets=None):
        # Gets the context batch matrix and returns logits array (vocab_size (64) size) for each index in the matrix.
        logits = self.token_embedding_table(context_ids) # Logit's Shape = Batch Size: B, Block Size: T, Vocab Size: C)

        if targets is None:
            return logits, None

        # Calculate the loss, based on the targets.

        B, T, C = logits.shape

        # Flattens the first two dimensions into one, so the shape goes from (Batch Size, Block Size, Vocab Size) to (Batch Size * Block Size, Vocab Size).
        #               Batch Item 1      Batch Item 2      Batch Item 3      Batch Item 4
        #               ________________  ________________  ________________  _________________
        # For example, [[[1, 2], [2, 3]], [[3, 4], [4, 5]], [[5, 6], [6, 7]], [[7, 8], [8, 9]]] (three nested arrays) becomes
        #               ______________  ______________  ______________  ______________
        #              [[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7], [7, 8], [8, 9]] (two nested arrays).
        # Essentially, the rows of all the batch items are joined into one flat list.
        # This is needed because the loss function expects the logits in the shape (Predictions, Vocab Size).
        logits = logits.view(B*T, C)

        # Converting the target matrix also into the same dimension as logits.
        targets = targets.view(B*T)

        loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, context_ids, max_new_tokens):
        # context_ids' shape = (B, T). Array of B Batch Items. Each item has T context ids.
        # context_ids = [[5, 2, 8], <== Batch 1
        #                [7, 1, 9]] <== Batch 2
        #                 ^  ^  ^
        #                 |  |  |
        #                 T  T  T
        #                 O  O  O
        #                 K  K  K
        #                 E  E  E
        #                 N  N  N
        #
        #                 1  2  3
        #
        # Shape: (2, 3)
        for _ in range(max_new_tokens):
            logits, _ = self(context_ids)

            # Extract logits of the last token index for each batch.
            # Example: [[0.1, -0.5,  0.3, ..., 0.2],  <== Batch 1; Logits for token id index 8
            #           [0.2,  0.1, -0.1, ..., 0.4]]  <== Batch 2; Logits for token id index 9
            last_token_index_logits = logits[:, -1, :]

            # Get the probabilities by applying softmax to all the logits of the last token index.
            # Example: [[0.02, 0.01, 0.05, ..., 0.03], <== Batch 1; Probabilities for each token id index, after token id index 8
            #           [0.03, 0.02, 0.01, ..., 0.04]] <== Batch 2; Probabilities for each token id index, after token id index 9
            probabilities = F.softmax(last_token_index_logits, dim=-1)

            # Returns indices of the next predicted token, for all the batches.
            # Example: [[42], <== Batch 1: Next token id index, after index 8
            #           [15]] <== Batch 2: Next token id index, after index 9
            indices_of_next_token = torch.multinomial(
                probabilities,
                # Randomly returns 1 token id index per batch based on the probability distribution.
                num_samples=1
            )

            # Append the predicted index for each batch at the end.
            # Updated: [[5, 2, 8, 42],
            #           [7, 1, 9, 15]]
            context_ids = torch.cat((context_ids, indices_of_next_token), dim=1)
        return context_ids

model = BigramLanguageModel(vocab_size)
m = model.to(device)

# Create an AdamW optimizer object (a specific optimization algorithm).
optimizer = torch.optim.AdamW(
                # Pass all model parameters (token_embedding_table which contains weights) to be updated during training.
                m.parameters(),
                # Learning rate: controls how much weights change at every update step.
                # Lower lr means slower but more stable learning while, Higher lr means faster but may diverge.
                # new_weight = old_weight - (learning_rate * gradient)
                lr=1e-3
            )

for steps in range(max_iters):
    # Every once in a while evaluate the loss on train and validation sets
    if steps % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {steps}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    context_ids, target_ids = get_batch('train')
    logits, loss = m(context_ids, target_ids)

    # Each cell in the token embedding table contains weights and each weights also has the property gradient.
    # The gradient is the derivative of loss with respect of weight. Hence, it indicates how much does the loss change by a minor change in weights.
    # The increase and decrease in the value of each weights is decided by this gradient.

    # We need to ensure that before we start calculating and storing the values in the gradient property of these weights, all the existing gradient properties in each weight of the token embedding table are cleared.
    # In short, it should not have the values calculate in the previous iteration.
    optimizer.zero_grad(set_to_none=True)

    # The loss goes back to each weight to calculate how much did that weight contributed into the loss.
    # Based on that calculates the gradient for that weights and stored the value in the gradient property of the weight.
    loss.backward()

    # This uses the calculated gradient value and update each weight cell in the token embedding table.
    optimizer.step()

def generate_next_tokens(context_ids, max_new_tokens=100):
    # Generate next tokens using the given context ids.
    generated_tokens = m.generate(context_ids, max_new_tokens)

    # Get the tokens for the first batch.
    generated_tokens_batch_1 = generated_tokens[0]

    # Convert the tokens array from tensor to list.
    token_id_index_list = generated_tokens_batch_1.tolist()

    # Use the tokens list to decode it using tokenizer.
    generated_text = decode(token_id_index_list)

    return generated_text

initial_context = torch.zeros((1, 1), dtype=torch.long)
generated_next_tokens = generate_next_tokens(initial_context, 500)
print(generated_next_tokens)

--2026-08-24 11:23:09--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.6’

input.txt.6         100%[===================>]   1.06M  --.-KB/s    in 0.1s    

2026-08-24 11:23:09 (9.10 MB/s) - ‘input.txt.6’ saved [1115394/1115394]

step 0: train loss 4.7305, val loss 4.7241
step 300: train loss 4.3818, val loss 4.3896
step 600: train loss 4.0801, val loss 4.0784
step 900: train loss 3.8066, val loss 3.8117
step 1200: train loss 3.5844, val loss 3.5850
step 1500: train loss 3.3757, val loss 3.3829
step 1800: train loss 3.2182, val loss 3.2218
step 2100: train loss 3.0817, val loss 3.0810
step 2400: train loss 2.9663, val l